# PediGuide — Colab trial run
Goal: prove training works on a free T4 before any real run.
Steps: check GPU → mount Drive → get code → short training → loss + sample → checkpoint to Drive.
Set Runtime → Change runtime type → **T4 GPU** first.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first (Runtime > Change runtime type > T4)'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/pediguide/checkpoints'
!mkdir -p "$CKPT_DIR"
print('checkpoints ->', CKPT_DIR)

## Get the code
Option A (repo): replace the URL with yours. Option B: upload the project folder manually, then `%cd` into it.

In [ ]:
YOUR_REPO_URL = 'https://github.com/reflexgc/AI-Chatbot-Train.git'  # <-- EDIT THIS
!git clone "$YOUR_REPO_URL" repo 2>/dev/null || echo 'clone skipped (edit URL or upload manually)'
%cd /content/repo 2>/dev/null || echo 'stay: upload files and %cd into the project folder'

In [ ]:
import json, torch
vocab = len(json.load(open('tokenizer/tokenizer.json', encoding='utf-8'))['vocab'])
tr = torch.load('data/formatted/train.bin', weights_only=True)
va = torch.load('data/formatted/val.bin', weights_only=True)
print(f'vocab={vocab} train_tokens={len(tr)} val_tokens={len(va)}')

## Trial training (~15-30 min on T4)
Small step count on purpose: this run proves the loop, the loss curve, and checkpointing — not final quality.

In [ ]:
!python train/train.py --steps 3000 --batch 32 --lr 3e-4 --eval-interval 250 --ckpt-interval 1000 --out "$CKPT_DIR/ckpt_trial.pt"

## What 'works' looks like
- `train` loss falls steadily (e.g. 9.0 → well below 5 within 3000 steps on this small corpus; exact numbers vary)
- `val` loss falls too (understanding, not just memorizing)
- A checkpoint file appears in Drive after step 1000 even if Colab disconnects

In [ ]:
import torch, json, sys
sys.path.insert(0, '.')
from model.model import GPTLanguageModel
from tokenizer.tokenizer import BPETokenizer
ckpt = torch.load(f'{CKPT_DIR}/ckpt_trial.pt', map_location='cpu')
print('step:', ckpt['step'], 'loss:', round(ckpt['loss'], 4))
tok = BPETokenizer.load('tokenizer/tokenizer.json')
model = GPTLanguageModel(vocab_size=ckpt['config']['vocab_size'])
model.load_state_dict(ckpt['model'])
model.eval()
prompt = '### Instruction: My 3-year-old has a runny nose. What could it be?\n### Response:'
ids = torch.tensor([tok.encode(prompt)])
with torch.no_grad():
    out = model.generate(ids, 80, temperature=0.8, top_k=50)
print(tok.decode(out[0].tolist()))

## Resume proof (disconnect survival)
Run this after any interruption: training continues from the Drive checkpoint instead of restarting.

In [ ]:
!python train/train.py --resume "$CKPT_DIR/ckpt_trial.pt" --steps 5000 --batch 32 --out "$CKPT_DIR/ckpt_trial.pt"